# Multimodal Skin-Lesion Classification — Image + Text (clinical metadata) Fusion

**Dataset:** PAD-UFES-20 (Kaggle `mahdavi1202/skin-cancer`) — 6 classes, smartphone photos + `metadata.csv`.

| Step | Model | Input |
|---|---|---|
| A | Image model — MobileNetV2 (transfer learning) | lesion photo |
| B | Text model — MLP on clinical metadata | age, itch, grew, hurt, changed, bleed, elevation, region |
| C | Late fusion — weighted average of A and B probabilities | both |
| D | **Feature-level fusion model** — A's image features + B's clinical features → joint classifier | both |

A and B are trained **separately** first. D is then built **from the two trained models** and fine-tuned.
All four are tested on the same held-out patients, then explained with **Grad-CAM**.

### How to run in Colab
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all` (about 15–20 min on a T4)
3. Allow Google Drive access when asked — models, plots and result tables are saved to `MyDrive/skin_multimodal/`.

In [ ]:
# ===================== CONFIGURATION (edit here) =====================
SEED          = 42
IMG_SIZE      = 224        # MobileNetV2 input size
BATCH_SIZE    = 32
SAVE_TO_DRIVE = True       # keep models/results in Google Drive (survives Colab resets)

# Model A - image model (two stages: frozen backbone, then fine-tune the top of the backbone)
IMG_HEAD_EPOCHS, IMG_HEAD_LR         = 10, 1e-3
IMG_FINETUNE_EPOCHS, IMG_FINETUNE_LR = 30, 5e-5
FINE_TUNE_FROM = "block_13_expand"   # MobileNetV2 layers from here to the end are fine-tuned

# Model B - text (clinical metadata) model
TXT_EPOCHS, TXT_LR = 200, 1e-3

# Model D - fusion model (two stages: frozen branches, then fine-tune everything except early backbone)
FUSION_HEAD_EPOCHS, FUSION_HEAD_LR         = 20, 1e-3
FUSION_FINETUNE_EPOCHS, FUSION_FINETUNE_LR = 15, 1e-5

KAGGLE_DATASET = "mahdavi1202/skin-cancer"   # PAD-UFES-20

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

import tensorflow as tf
import keras
from keras import layers

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_recall_fscore_support, roc_auc_score, confusion_matrix)
import joblib

keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE  ->  Runtime > Change runtime type > T4 GPU (CPU training is very slow)")

OUT_DIR, CKPT_DIR = "outputs", "checkpoints"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = "/content/drive/MyDrive/skin_multimodal"
    except ImportError:
        print("Not running in Colab - saving to ./outputs instead")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print("Saving results to:", OUT_DIR)

## 1. Load the two files: lesion images + `metadata.csv` (text)

Same cleaning as the EDA notebook: the 19 rows with `UNK` in itch / hurt / bleed / elevation are dropped.
`grew` and `changed` have ~400 `UNK` values each, so for them `UNK` is kept as its own category.

In [ ]:
import kagglehub
DATA_ROOT = kagglehub.dataset_download(KAGGLE_DATASET)   # cached by Colab after the first download

meta = pd.read_csv(os.path.join(DATA_ROOT, "metadata.csv"))
path_map = {os.path.basename(p): p for p in glob.glob(os.path.join(DATA_ROOT, "**", "*.png"), recursive=True)}
meta["path"] = meta["img_id"].map(path_map)
print(f"metadata rows: {len(meta)} | images found: {meta['path'].notna().sum()}")

SYMPTOM_COLS = ["itch", "grew", "hurt", "changed", "bleed", "elevation"]
TEXT_COLS = ["age"] + SYMPTOM_COLS + ["region"]          # the clinical "text" inputs
for c in SYMPTOM_COLS + ["region"]:
    meta[c] = meta[c].astype(str).str.upper().str.strip()

keep = meta[["itch", "hurt", "bleed", "elevation"]].ne("UNK").all(axis=1) & meta["path"].notna()
data = meta.loc[keep].reset_index(drop=True)

CLASS_NAMES = sorted(data["diagnostic"].unique())
CLASS_FULL = {"ACK": "Actinic Keratosis", "BCC": "Basal Cell Carcinoma", "MEL": "Melanoma",
              "NEV": "Nevus", "SCC": "Squamous Cell Carcinoma", "SEK": "Seborrheic Keratosis"}
NUM_CLASSES = len(CLASS_NAMES)
data["label"] = data["diagnostic"].map({c: i for i, c in enumerate(CLASS_NAMES)})
y_all = data["label"].to_numpy()
Y_all = keras.utils.to_categorical(y_all, NUM_CLASSES).astype("float32")   # one-hot labels

print(f"rows used: {len(data)} | classes: {CLASS_NAMES}")
data[["img_id", "patient_id", "diagnostic"] + TEXT_COLS].head()

## 2. Patient-level train / validation / test split

PAD-UFES-20 has several photos per patient. If photos of one patient land in both train and test, the test score is inflated (leakage).
We reuse the **same 5 patient-grouped, class-stratified folds as the EDA notebook** (`splits.csv`):
fold 0 → **test**, fold 1 → **validation**, folds 2–4 → **training** (≈ 60 / 20 / 20).
All four models use exactly these splits, so their scores are directly comparable.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
data["fold"] = -1
for k, (_, fold_idx) in enumerate(sgkf.split(np.zeros(len(data)), data["diagnostic"], groups=data["patient_id"])):
    data.loc[fold_idx, "fold"] = k
data["split"] = data["fold"].map(lambda k: "test" if k == 0 else ("val" if k == 1 else "train"))

tr_idx = np.where(data["split"] == "train")[0]
va_idx = np.where(data["split"] == "val")[0]
te_idx = np.where(data["split"] == "test")[0]

patients = {s: set(data.loc[data["split"] == s, "patient_id"]) for s in ("train", "val", "test")}
assert not (patients["train"] & patients["val"]), "patient leakage train/val"
assert not (patients["train"] & patients["test"]), "patient leakage train/test"
assert not (patients["val"] & patients["test"]), "patient leakage val/test"
print(f"train {len(tr_idx)} | val {len(va_idx)} | test {len(te_idx)} images  (no patient shared between splits)")

data[["img_id", "patient_id", "diagnostic", "fold", "split"]].to_csv(f"{OUT_DIR}/splits.csv", index=False)
pd.crosstab(data["split"], data["diagnostic"], margins=True).loc[["train", "val", "test", "All"]]

## 3. Prepare the inputs

**Images:** every photo is resized to 224×224 once and kept in memory (fast training).
Random flips / rotation / zoom / shift / brightness / contrast are applied to **training** images only, which also covers lighting and camera-angle variation.

**Text (clinical metadata):** age is standardised; each symptom (TRUE / FALSE / UNK) and the body region are one-hot encoded.
The encoder is fitted on the training split only and saved for the app.

**Class imbalance:** BCC has ~16× more images than MEL. The EDA notebook tried SMOTE on the table, but SMOTE creates synthetic rows that have no photo,
so it cannot be used for the image or fusion models. All models therefore use **class-weighted loss** (rare classes count more), which keeps the comparison fair.

In [ ]:
def load_image(path, size=IMG_SIZE):
    """Read one photo as a size x size RGB uint8 array."""
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB").resize((size, size), Image.BILINEAR), dtype=np.uint8)

cache = f"cache/images_{IMG_SIZE}.npy"
if os.path.exists(cache) and np.load(cache, mmap_mode="r").shape[0] == len(data):
    X_img = np.load(cache)
else:
    with ThreadPoolExecutor(max_workers=16) as pool:
        X_img = np.stack(list(pool.map(load_image, data["path"])))
    os.makedirs("cache", exist_ok=True)
    np.save(cache, X_img)
print("images:", X_img.shape, X_img.dtype, f"{X_img.nbytes / 1e6:.0f} MB")

In [ ]:
preproc = ColumnTransformer([
    ("age", StandardScaler(), ["age"]),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), SYMPTOM_COLS + ["region"]),
])
preproc.fit(data.loc[tr_idx, TEXT_COLS])                     # fit on training patients only
X_meta = preproc.transform(data[TEXT_COLS]).astype("float32")
META_FEATURES = [str(f) for f in preproc.get_feature_names_out()]
joblib.dump(preproc, f"{OUT_DIR}/clinical_preprocessor.joblib")
print(f"{X_meta.shape[1]} clinical features:", META_FEATURES)

cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_all[tr_idx])
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}
pd.DataFrame({"train images": np.bincount(y_all[tr_idx], minlength=NUM_CLASSES),
              "class weight": cw.round(2)}, index=CLASS_NAMES)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
AUGMENT_LAYERS = [
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15, fill_mode="reflect"),
    layers.RandomZoom(0.15, fill_mode="reflect"),
    layers.RandomTranslation(0.1, 0.1, fill_mode="reflect"),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
]

def augment_images(images):
    for layer in AUGMENT_LAYERS:
        images = layer(images, training=True)
    return tf.clip_by_value(images, 0.0, 255.0)

def make_dataset(idx, inputs, training=False):
    """inputs = 'image' (Model A), 'meta' (Model B) or 'both' (Model D)."""
    if inputs == "image":
        x = X_img[idx]
    elif inputs == "meta":
        x = X_meta[idx]
    else:
        x = {"image": X_img[idx], "meta": X_meta[idx]}
    ds = tf.data.Dataset.from_tensor_slices((x, Y_all[idx]))
    if training:
        ds = ds.shuffle(len(idx), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)

    def prepare(x, y):
        if inputs == "meta":
            return x, y
        img = tf.cast(x["image"] if inputs == "both" else x, tf.float32)
        if training:
            img = augment_images(img)
        return ({"image": img, "meta": x["meta"]} if inputs == "both" else img), y

    return ds.map(prepare, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# quick look at what the image model sees during training
sample = next(iter(make_dataset(tr_idx[:8], "image", training=True)))[0].numpy().astype("uint8")
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for ax, im in zip(axes, sample):
    ax.imshow(im); ax.axis("off")
fig.suptitle("Augmented training images"); plt.show()

### Shared training and evaluation helpers

* **Loss:** categorical cross-entropy (+ class weights, + light label smoothing).
* **Model selection:** the epoch with the best **validation macro-F1** is kept (macro-F1 treats every class equally, so the rare classes matter as much as BCC).
* **Test metrics:** accuracy, balanced accuracy, macro precision / recall / F1, weighted F1, macro ROC-AUC,
  plus per-class precision, recall (sensitivity), F1 and **false-positive rate**, and the confusion matrix.

In [ ]:
HEAD_PREFIXES = ("img_", "meta_", "fusion_")   # every layer we add is named with one of these prefixes

def is_backbone(layer):
    return not isinstance(layer, keras.layers.InputLayer) and not layer.name.startswith(HEAD_PREFIXES)

def set_backbone_trainable(model, from_layer=None):
    """Freeze MobileNetV2. If from_layer is given, unfreeze it and every later backbone layer
    (BatchNorm layers stay frozen - standard practice when fine-tuning on a small dataset)."""
    assert from_layer is None or from_layer in [l.name for l in model.layers], f"{from_layer} not found"
    unfreeze = False
    for layer in model.layers:
        if not is_backbone(layer):
            continue
        if layer.name == from_layer:
            unfreeze = True
        layer.trainable = unfreeze and not isinstance(layer, layers.BatchNormalization)

def count_params(model):
    t = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    f = sum(int(np.prod(w.shape)) for w in model.non_trainable_weights)
    print(f"{model.name}: {t:,} trainable / {f:,} frozen parameters")

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
                  metrics=["accuracy", keras.metrics.F1Score(average="macro", name="macro_f1")])

def fit_stage(model, train_ds, val_ds, epochs, ckpt_path, patience, best_so_far=None):
    """Train one stage. The best weights (by val macro-F1, across all stages) are kept in ckpt_path."""
    callbacks = [
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_macro_f1", mode="max", save_best_only=True,
                                        save_weights_only=True, initial_value_threshold=best_so_far),
        keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max", patience=patience),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ]
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, class_weight=CLASS_WEIGHT,
                     callbacks=callbacks, verbose=2)
    best = max(hist.history["val_macro_f1"])
    return hist.history, best if best_so_far is None else max(best, best_so_far)

def plot_history(histories, title, key):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for ax, m in zip(axes, ["loss", "accuracy", "macro_f1"]):
        tr = sum((h[m] for h in histories), [])
        va = sum((h["val_" + m] for h in histories), [])
        ax.plot(tr, label="train"); ax.plot(va, label="validation")
        if len(histories) > 1:
            ax.axvline(len(histories[0][m]) - 0.5, ls="--", c="gray", label="fine-tuning starts")
        ax.set_title(m); ax.set_xlabel("epoch"); ax.grid(alpha=0.3); ax.legend()
    fig.suptitle(title); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/history_{key}.png", dpi=150, bbox_inches="tight"); plt.show()

RESULTS, PER_CLASS, VAL_PROBS, TEST_PROBS = {}, {}, {}, {}

def evaluate(title, key, probs, idx=None):
    """Test-set report for one model: summary metrics, per-class table, confusion matrix. Saves everything."""
    idx = te_idx if idx is None else idx
    y_true, y_pred = y_all[idx], probs.argmax(axis=1)
    labels = list(range(NUM_CLASSES))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fp = cm.sum(axis=0) - np.diag(cm)
    tn = cm.sum() - cm.sum(axis=0) - cm.sum(axis=1) + np.diag(cm)
    try:
        auc = roc_auc_score(y_true, probs, multi_class="ovr", average="macro", labels=labels)
    except ValueError:
        auc = float("nan")
    RESULTS[title] = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro precision": p.mean(), "Macro recall": r.mean(), "Macro F1": f.mean(),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted"),
        "Macro AUC": auc,
    }
    per_class = pd.DataFrame({"precision": p, "recall (sensitivity)": r, "F1": f,
                              "false-positive rate": fp / np.maximum(fp + tn, 1), "support": s}, index=CLASS_NAMES)
    PER_CLASS[title] = per_class
    per_class.to_csv(f"{OUT_DIR}/per_class_{key}.csv")
    preds = pd.DataFrame(probs, columns=CLASS_NAMES)
    preds.insert(0, "img_id", data["img_id"].to_numpy()[idx])
    preds.insert(1, "true", [CLASS_NAMES[i] for i in y_true])
    preds.insert(2, "predicted", [CLASS_NAMES[i] for i in y_pred])
    preds.to_csv(f"{OUT_DIR}/test_predictions_{key}.csv", index=False)

    print(f"===== {title} - TEST SET ({len(idx)} images) =====")
    display(pd.DataFrame([RESULTS[title]], index=[title]).round(3))
    display(per_class.round(3))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
    axes[0].set_title("Confusion matrix (counts)")
    sns.heatmap(cm / np.maximum(cm.sum(axis=1, keepdims=True), 1), annot=True, fmt=".2f", cmap="Blues",
                vmin=0, vmax=1, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
    axes[1].set_title("Confusion matrix (row-normalised = recall)")
    for ax in axes:
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    fig.suptitle(title); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/confusion_{key}.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Model A — Image model (MobileNetV2, transfer learning)

`photo → MobileNetV2 (ImageNet weights) → global average pooling → Dense 256 (image embedding) → softmax (6 classes)`

* **Stage 1:** backbone frozen, only the new head is trained.
* **Stage 2:** the last MobileNetV2 blocks (from `block_13_expand`) are unfrozen and fine-tuned with a small learning rate.

In [ ]:
def build_image_model():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = layers.Rescaling(1 / 127.5, offset=-1, name="img_preprocess")(inp)   # pixels -> [-1, 1], as MobileNetV2 expects
    backbone = keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                                              input_tensor=x, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.GlobalAveragePooling2D(name="img_gap")(backbone.output)        # 1280 features
    x = layers.Dropout(0.3, name="img_dropout1")(x)
    x = layers.Dense(256, activation="relu", name="img_embedding")(x)          # used later by the fusion model
    x = layers.Dropout(0.3, name="img_dropout2")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="img_output")(x)
    return keras.Model(inp, out, name="image_model")

image_model = build_image_model()
img_train_ds = make_dataset(tr_idx, "image", training=True)
img_val_ds, img_test_ds = make_dataset(va_idx, "image"), make_dataset(te_idx, "image")

In [ ]:
ckpt = f"{CKPT_DIR}/image_model.weights.h5"

print("--- Stage 1: train the head, backbone frozen ---")
set_backbone_trainable(image_model)
compile_model(image_model, IMG_HEAD_LR); count_params(image_model)
h1, best = fit_stage(image_model, img_train_ds, img_val_ds, IMG_HEAD_EPOCHS, ckpt, patience=4)

print(f"--- Stage 2: fine-tune from {FINE_TUNE_FROM} ---")
set_backbone_trainable(image_model, FINE_TUNE_FROM)
compile_model(image_model, IMG_FINETUNE_LR); count_params(image_model)
h2, best = fit_stage(image_model, img_train_ds, img_val_ds, IMG_FINETUNE_EPOCHS, ckpt, patience=8, best_so_far=best)

image_model.load_weights(ckpt)                       # best epoch over both stages
image_model.save(f"{OUT_DIR}/image_model.keras")
print(f"Best validation macro-F1: {best:.3f}  ->  saved image_model.keras")
plot_history([h1, h2], "Model A - Image (MobileNetV2)", "image")

In [ ]:
VAL_PROBS["image"] = image_model.predict(img_val_ds, verbose=0)
TEST_PROBS["image"] = image_model.predict(img_test_ds, verbose=0)
evaluate("A: Image (MobileNetV2)", "image", TEST_PROBS["image"])

## 5. Model B — Text model (clinical metadata from `metadata.csv`)

`clinical features → Dense 64 → BatchNorm → Dense 32 (clinical embedding) → softmax (6 classes)`

A small network is enough: the input is ~30 numbers per patient.

In [ ]:
def build_text_model(n_features):
    inp = keras.Input((n_features,), name="meta")
    x = layers.Dense(64, activation="relu", name="meta_dense1")(inp)
    x = layers.BatchNormalization(name="meta_bn1")(x)
    x = layers.Dropout(0.3, name="meta_dropout1")(x)
    x = layers.Dense(32, activation="relu", name="meta_embedding")(x)    # used later by the fusion model
    x = layers.Dropout(0.3, name="meta_dropout2")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="meta_output")(x)
    return keras.Model(inp, out, name="text_model")

text_model = build_text_model(X_meta.shape[1])
txt_train_ds = make_dataset(tr_idx, "meta", training=True)
txt_val_ds, txt_test_ds = make_dataset(va_idx, "meta"), make_dataset(te_idx, "meta")

ckpt = f"{CKPT_DIR}/text_model.weights.h5"
compile_model(text_model, TXT_LR); count_params(text_model)
h_txt, best = fit_stage(text_model, txt_train_ds, txt_val_ds, TXT_EPOCHS, ckpt, patience=25)
text_model.load_weights(ckpt)
text_model.save(f"{OUT_DIR}/text_model.keras")
print(f"Best validation macro-F1: {best:.3f}  ->  saved text_model.keras")
plot_history([h_txt], "Model B - Text / clinical metadata (MLP)", "text")

In [ ]:
VAL_PROBS["text"] = text_model.predict(txt_val_ds, verbose=0)
TEST_PROBS["text"] = text_model.predict(txt_test_ds, verbose=0)
evaluate("B: Text (clinical MLP)", "text", TEST_PROBS["text"])

## 6. Model C — Late fusion (decision level)

`P = w · P_image + (1 − w) · P_text`. The weight `w` is chosen on the **validation** set, then applied once to the test set.
This is the simple baseline the feature-level fusion model has to beat.

In [ ]:
weights = np.round(np.arange(0, 1.0001, 0.05), 2)
val_f1 = [f1_score(y_all[va_idx], (w * VAL_PROBS["image"] + (1 - w) * VAL_PROBS["text"]).argmax(1), average="macro")
          for w in weights]
LATE_W = float(weights[int(np.argmax(val_f1))])
print(f"Best image weight w = {LATE_W}  (validation macro-F1 {max(val_f1):.3f})")

plt.figure(figsize=(7, 3.5))
plt.plot(weights, val_f1, marker="o"); plt.axvline(LATE_W, ls="--", c="gray")
plt.xlabel("w (weight of image model)"); plt.ylabel("validation macro-F1")
plt.title("Late fusion: choosing w  (0 = text only, 1 = image only)"); plt.grid(alpha=0.3)
plt.savefig(f"{OUT_DIR}/late_fusion_weight.png", dpi=150, bbox_inches="tight"); plt.show()

TEST_PROBS["late"] = LATE_W * TEST_PROBS["image"] + (1 - LATE_W) * TEST_PROBS["text"]
evaluate("C: Late fusion", "late_fusion", TEST_PROBS["late"])

## 7. Model D — Feature-level multimodal fusion model

```
photo ──► trained Model A (MobileNetV2) ──► 256-d image embedding ─┐
                                                                    ├─► concat (288) ─► BatchNorm ─► Dense 128 ─► Dropout ─► softmax (6)
clinical ─► trained Model B (MLP) ──────────► 32-d clinical embedding ┘
```

Both branches are loaded from the saved Models A and B (so they start from what they already learned).
* **Stage 1:** both branches frozen, only the fusion head is trained.
* **Stage 2:** branches unfrozen (backbone from `block_13_expand`) and the whole model is fine-tuned with a very small learning rate.

In [ ]:
img_branch = keras.models.load_model(f"{OUT_DIR}/image_model.keras", compile=False)
txt_branch = keras.models.load_model(f"{OUT_DIR}/text_model.keras", compile=False)

img_feat = img_branch.get_layer("img_embedding").output     # 256-d
txt_feat = txt_branch.get_layer("meta_embedding").output    # 32-d
x = layers.Concatenate(name="fusion_concat")([img_feat, txt_feat])
x = layers.BatchNormalization(name="fusion_bn")(x)           # puts both modalities on the same scale
x = layers.Dense(128, activation="relu", name="fusion_dense")(x)
x = layers.Dropout(0.4, name="fusion_dropout")(x)
out = layers.Dense(NUM_CLASSES, activation="softmax", name="fusion_output")(x)
fusion_model = keras.Model({"image": img_branch.input, "meta": txt_branch.input}, out, name="fusion_model")

def set_fusion_trainable(model, finetune):
    for layer in model.layers:
        if layer.name.startswith("fusion_"):
            layer.trainable = True
        elif layer.name.startswith(("img_", "meta_")):
            layer.trainable = finetune
    set_backbone_trainable(model, FINE_TUNE_FROM if finetune else None)

fus_train_ds = make_dataset(tr_idx, "both", training=True)
fus_val_ds, fus_test_ds = make_dataset(va_idx, "both"), make_dataset(te_idx, "both")

In [ ]:
ckpt = f"{CKPT_DIR}/fusion_model.weights.h5"

print("--- Stage 1: train the fusion head, both branches frozen ---")
set_fusion_trainable(fusion_model, finetune=False)
compile_model(fusion_model, FUSION_HEAD_LR); count_params(fusion_model)
f1_hist, best = fit_stage(fusion_model, fus_train_ds, fus_val_ds, FUSION_HEAD_EPOCHS, ckpt, patience=6)

print("--- Stage 2: fine-tune the whole fusion model ---")
set_fusion_trainable(fusion_model, finetune=True)
compile_model(fusion_model, FUSION_FINETUNE_LR); count_params(fusion_model)
f2_hist, best = fit_stage(fusion_model, fus_train_ds, fus_val_ds, FUSION_FINETUNE_EPOCHS, ckpt, patience=5, best_so_far=best)

fusion_model.load_weights(ckpt)
fusion_model.save(f"{OUT_DIR}/fusion_model.keras")
print(f"Best validation macro-F1: {best:.3f}  ->  saved fusion_model.keras")
plot_history([f1_hist, f2_hist], "Model D - Feature-level fusion", "fusion")

In [ ]:
VAL_PROBS["fusion"] = fusion_model.predict(fus_val_ds, verbose=0)
TEST_PROBS["fusion"] = fusion_model.predict(fus_test_ds, verbose=0)
evaluate("D: Feature fusion", "fusion", TEST_PROBS["fusion"])

## 8. Compare all models (same test patients)

In [ ]:
comparison = pd.DataFrame(RESULTS).T
comparison.to_csv(f"{OUT_DIR}/model_comparison.csv")
display(comparison.round(3))

cols = ["Accuracy", "Balanced accuracy", "Macro F1", "Macro AUC"]
ax = comparison[cols].plot.bar(figsize=(13, 4.5), rot=0, width=0.8)
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", fontsize=7)
ax.set_ylim(0, 1.05); ax.set_ylabel("test score"); ax.grid(axis="y", alpha=0.3)
ax.legend(loc="lower right", ncol=4); ax.set_title("Image vs Text vs Fusion")
plt.savefig(f"{OUT_DIR}/model_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

recall_table = pd.DataFrame({m: t["recall (sensitivity)"] for m, t in PER_CLASS.items()})
recall_table["test images"] = PER_CLASS[next(iter(PER_CLASS))]["support"]
recall_table.to_csv(f"{OUT_DIR}/per_class_recall_comparison.csv")
print("Per-class recall (sensitivity) - how many of each disease each model catches:")
display(recall_table.round(3))

## 9. Grad-CAM — which part of the photo drove the prediction

Grad-CAM takes the gradient of the predicted class score with respect to the last MobileNetV2 feature map (`out_relu`, 7×7),
and highlights the regions that pushed the prediction up (red = strong influence).
Shown for one test image per class, for the image-only model and the fusion model.

In [ ]:
def make_gradcam_model(model, layer_name="out_relu"):
    names = [n for n in ("image", "meta") if n in [l.name for l in model.layers]]
    inputs = {n: model.get_layer(n).output for n in names}
    inputs = inputs["image"] if names == ["image"] else inputs
    return keras.Model(inputs, [model.get_layer(layer_name).output, model.output])

def grad_cam(gc_model, inputs, class_idx=None):
    with tf.GradientTape() as tape:
        conv, preds = gc_model(inputs, training=False)
        if class_idx is None:
            class_idx = int(tf.argmax(preds[0]))
        score = preds[:, class_idx]
    grads = tape.gradient(score, conv)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)          # importance of each feature map
    cam = tf.nn.relu(tf.reduce_sum(weights * conv, axis=-1))[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE))[..., 0].numpy()
    return cam, preds[0].numpy()

gc_image, gc_fusion = make_gradcam_model(image_model), make_gradcam_model(fusion_model)

picks = [te_idx[y_all[te_idx] == c][0] for c in range(NUM_CLASSES) if (y_all[te_idx] == c).any()]
fig, axes = plt.subplots(len(picks), 3, figsize=(11, 3.6 * len(picks)), squeeze=False)
for row, i in enumerate(picks):
    img = X_img[i].astype("float32")[None]
    cam_a, p_a = grad_cam(gc_image, img)
    cam_d, p_d = grad_cam(gc_fusion, {"image": img, "meta": X_meta[i][None]})
    axes[row, 0].imshow(X_img[i])
    axes[row, 0].set_title(f"True: {CLASS_NAMES[y_all[i]]} ({CLASS_FULL.get(CLASS_NAMES[y_all[i]], '')})", fontsize=9)
    for ax, cam, p, name in [(axes[row, 1], cam_a, p_a, "Image model"), (axes[row, 2], cam_d, p_d, "Fusion model")]:
        ax.imshow(X_img[i]); ax.imshow(cam, cmap="jet", alpha=0.4)
        ax.set_title(f"{name}: {CLASS_NAMES[p.argmax()]} ({p.max():.0%})", fontsize=9)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/gradcam_examples.png", dpi=150, bbox_inches="tight"); plt.show()

## 10. Predict a single case (demo)

`predict_case(...)` takes one photo + the clinical answers and returns the fusion model's probabilities with a Grad-CAM map.
Symptoms accept `True` / `False` / `None` (unknown). This is the same function the UI will call.

In [ ]:
def to_flag(v):
    if v is None:
        return "UNK"
    if isinstance(v, str):
        return v.strip().upper()
    return "TRUE" if v else "FALSE"

def predict_case(image_path, age, region, itch, grew, hurt, changed, bleed, elevation, show=True):
    img = load_image(image_path).astype("float32")[None]
    row = pd.DataFrame([{"age": age, "region": str(region).strip().upper(),
                         "itch": to_flag(itch), "grew": to_flag(grew), "hurt": to_flag(hurt),
                         "changed": to_flag(changed), "bleed": to_flag(bleed), "elevation": to_flag(elevation)}])
    meta_vec = preproc.transform(row[TEXT_COLS]).astype("float32")
    cam, probs = grad_cam(gc_fusion, {"image": img, "meta": meta_vec})
    if show:
        order = probs.argsort()
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
        axes[0].imshow(img[0].astype("uint8")); axes[0].set_title("Input photo")
        axes[1].imshow(img[0].astype("uint8")); axes[1].imshow(cam, cmap="jet", alpha=0.4)
        axes[1].set_title("Grad-CAM (fusion model)")
        for ax in axes[:2]:
            ax.axis("off")
        axes[2].barh([f"{CLASS_NAMES[k]} - {CLASS_FULL.get(CLASS_NAMES[k], '')}" for k in order], probs[order])
        axes[2].set_xlim(0, 1); axes[2].set_title(f"Prediction: {CLASS_NAMES[probs.argmax()]} ({probs.max():.0%})")
        plt.tight_layout(); plt.show()
    return dict(zip(CLASS_NAMES, probs.round(4).tolist()))

r = data.iloc[te_idx[0]]                     # an unseen test patient
print("Ground truth:", r["diagnostic"], "| clinical:", r[TEXT_COLS].to_dict())
predict_case(r["path"], age=r["age"], region=r["region"], itch=r["itch"], grew=r["grew"], hurt=r["hurt"],
             changed=r["changed"], bleed=r["bleed"], elevation=r["elevation"])

## 11. Saved files

Everything needed for the app / report is in `MyDrive/skin_multimodal/`:
`image_model.keras`, `text_model.keras`, `fusion_model.keras`, `clinical_preprocessor.joblib`, `config.json`,
`model_comparison.csv/.png`, per-class tables, confusion matrices, training curves, Grad-CAM examples and test predictions.

Reload later (e.g. in Streamlit):
```python
fusion_model = keras.models.load_model("fusion_model.keras")
preproc = joblib.load("clinical_preprocessor.joblib")
config = json.load(open("config.json"))
```

In [ ]:
config = {
    "dataset": "PAD-UFES-20 (Kaggle mahdavi1202/skin-cancer)",
    "class_names": CLASS_NAMES,
    "class_full_names": CLASS_FULL,
    "img_size": IMG_SIZE,
    "image_preprocessing": "RGB, resized to img_size x img_size, raw 0-255 floats (scaling is inside the model)",
    "text_columns": TEXT_COLS,
    "symptom_values": ["TRUE", "FALSE", "UNK"],
    "clinical_features": META_FEATURES,
    "late_fusion_image_weight": LATE_W,
    "gradcam_layer": "out_relu",
    "models": {"image": "image_model.keras", "text": "text_model.keras", "fusion": "fusion_model.keras"},
    "preprocessor": "clinical_preprocessor.joblib",
    "test_results": {m: {k: round(float(v), 4) for k, v in r.items()} for m, r in RESULTS.items()},
}
with open(f"{OUT_DIR}/config.json", "w") as fh:
    json.dump(config, fh, indent=2)

for name in sorted(os.listdir(OUT_DIR)):
    print(f"{name:45s} {os.path.getsize(os.path.join(OUT_DIR, name)) / 1e6:8.2f} MB")

### Notes for the Phase II report
* **Dataset change:** Phase I planned HAM10000 (7 dermoscopic classes). This notebook uses PAD-UFES-20 — **6 classes, smartphone (clinical) photos** —
  because it is the dataset that actually contains the patient-history fields (itch, hurt, grew, changed, bleed, elevation). Update the report text accordingly.
* **Small test set:** ~450 test images, and only a handful of melanomas (MEL). Per-class MEL numbers can swing a lot between runs; report them with that caveat.
* **Run-to-run variation:** GPU training is not bit-for-bit deterministic, so expect scores to move by a point or two if you re-run.